# Import Library

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
print("LOAD DATA BERSIH")
df = pd.read_csv('../data/processed/clean_data_games.csv')

df['tags_clean'] = df['tags_clean'].fillna('')
display(df[['app_id', 'title', 'tags_clean']].head())

LOAD DATA BERSIH


,app_id,title,tags_clean
0,13500,Prince of Persia: Warrior Within™,Action Adventure Parkour Third Person Great So...
1,22364,BRINK: Agents of Change,Action
2,113020,Monaco: What's Yours Is Mine,Co-op Stealth Indie Heist Local Co-Op Strategy...
3,226560,Escape Dead Island,Zombies Adventure Survival Action Third Person...
4,249050,Dungeon of the ENDLESS™,Roguelike Strategy Tower Defense Pixel Graphic...


# Proses TF-IDF (Term Frequency-Inverse Document Frequency)

In [5]:

print("MEMBUAT MATRIKS TF-IDF")
tfidf = TfidfVectorizer()

# Mengubah kolom tags_clean menjadi matriks angka
tfidf_matrix = tfidf.fit_transform(df['tags_clean'])

print(f"Ukuran Matriks TF-IDF: {tfidf_matrix.shape}")

MEMBUAT MATRIKS TF-IDF
Ukuran Matriks TF-IDF: (50872, 482)


## Fungsi rekomendasi

In [6]:
def get_recommendation(game_title, df, tfidf_matrix, top_n=5):
    # 1. Cari index game yang di-input user
    try:
        idx = df[df['title'].str.lower() == game_title.lower()].index[0]
    except IndexError:
        return f"Game '{game_title}' tidak ditemukan di database."

    # 2. Ambil vektor TF-IDF khusus dari game tersebut
    game_vector = tfidf_matrix[idx]

    # 3. Hitung Cosine Similarity game ini dengan game lain
    sim_scores = cosine_similarity(game_vector, tfidf_matrix).flatten()

    # 4. Urutkan dari yang paling mirip
    similar_indices = sim_scores.argsort()[::-1]

    # 5. Top N game
    top_indices = similar_indices[1:top_n+1]

    # 6. Tampilkan hasil
    recom_df = df.iloc[top_indices][['title', 'tags_clean', 'rating']].copy()
    recom_df['similarity_score'] = sim_scores[top_indices] 
    
    return recom_df

# Test rekomendasi

In [9]:
print("Mencari rekomendasi untuk 'Left 4 Dead 2'...")
display(get_recommendation('Left 4 Dead 2', df, tfidf_matrix, top_n=5))

Mencari rekomendasi untuk 'Left 4 Dead 2'...


,title,tags_clean,rating,similarity_score
50870,Forgive Me Father 2,Early Access FPS Action Retro First-Person Lov...,Very Positive,0.0
50869,Eternights,,Very Positive,0.0
50868,PAYDAY 3,,Mostly Negative,0.0
50867,I Expect You To Die 3: Cog in the Machine,,Very Positive,0.0
50866,Train Sim World® 4,,Mixed,0.0


In [10]:
print("Mencari rekomendasi untuk 'Stardew Valley'...")
display(get_recommendation('Stardew Valley', df, tfidf_matrix, top_n=5))

Mencari rekomendasi untuk 'Stardew Valley'...


,title,tags_clean,rating,similarity_score
50870,Forgive Me Father 2,Early Access FPS Action Retro First-Person Lov...,Very Positive,0.0
50869,Eternights,,Very Positive,0.0
50868,PAYDAY 3,,Mostly Negative,0.0
50867,I Expect You To Die 3: Cog in the Machine,,Very Positive,0.0
50866,Train Sim World® 4,,Mixed,0.0


In [12]:
cek_game = df[df['title'].str.contains('Left 4 Dead 2|Stardew Valley', case=False, na=False)]
display(cek_game[['app_id', 'title', 'tags_clean']])

,app_id,title,tags_clean
12711,550,Left 4 Dead 2,
13113,440820,Stardew Valley Soundtrack,Simulation RPG Indie Soundtrack Music Great So...
14459,413150,Stardew Valley,
43656,1174581,Dying Light - Left 4 Dead 2 Weapon Pack,Action RPG Gore Violent Multiplayer Zombies


In [13]:
# Cek banyak game yang tag-nya kosong
jumlah_kosong = len(df[df['tags_clean'] == ''])
print(f"Total game dengan tag kosong: {jumlah_kosong} dari {len(df)} game")

Total game dengan tag kosong: 1244 dari 50872 game


In [ ]:
# Isi manual tag
df.loc[df['title'] == 'Stardew Valley', 'tags_clean'] = 'Farming Simulation RPG Pixel Graphics Multiplayer'
df.loc[df['title'] == 'Left 4 Dead 2', 'tags_clean'] = 'Zombies Co-op Shooter Action Multiplayer First-Person'

In [15]:
# buang game dengan tag kosong
df = df[df['tags_clean'] != '']

df = df.reset_index(drop=True)

tfidf_matrix = tfidf.fit_transform(df['tags_clean'])

print(f"Pembersihan selesai. Sisa data sekarang: {len(df)} game.")
print(f"Ukuran Matriks TF-IDF baru: {tfidf_matrix.shape}")

Pembersihan selesai. Sisa data sekarang: 49630 game.
Ukuran Matriks TF-IDF baru: (49630, 482)


In [17]:
print("Mencari rekomendasi untuk 'Stardew Valley'")
display(get_recommendation('Stardew Valley', df, tfidf_matrix, top_n=5))

Mencari rekomendasi untuk 'Stardew Valley'


,title,tags_clean,rating,similarity_score
34343,Farming Simulator 2013 - Classics,Simulation Casual Farming Multiplayer,Positive,0.772893
3786,Graveyard Keeper - Game Of Crone,Simulation RPG Adventure Indie Pixel Graphics ...,Mixed,0.745089
1547,Farming Simulator 19 - Alpine Farming Expansion,Simulation Farming,Mixed,0.735789
24507,Our Love Will Grow,Farming Sim Simulation Adventure RPG Indie RPG...,Mixed,0.713183
2691,Gleaner Heights,Farming Sim RPG Indie Simulation Action Advent...,Mostly Positive,0.671696


# export model

In [18]:
import pickle
import os

os.makedirs('../models', exist_ok=True)

with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('../models/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

df.to_pickle('../models/clean_games_df.pkl')

print("Berhasil menyimpan model")

Berhasil menyimpan model
